# High-Performance Dual-GPU Parallel SBD Training
**Task Parallelism on 2x NVIDIA Tesla T4 GPUs:**
- **GPU 0 (`cuda:0`):** Bidirectional CharLM + Weighted Cross-Entropy Loss ($pos\_weight=4.0$).
- **GPU 1 (`cuda:1`):** Bidirectional CharLM + Focal Cross-Entropy Loss ($\gamma=2.0, pos\_weight=4.0$).
- Fully asynchronous multiprocessing with 100% dual-GPU utilization.
- Joint ensembling and evaluation on Chagatai test set (295 sequential samples).


In [ ]:
import os
import sys
import json
import time
import shutil
import subprocess
import importlib.util
from pathlib import Path

for pkg in ['stanza', 'datasets', 'huggingface_hub']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)

import torch
from datasets import load_dataset

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"Detected GPU Count: {device_count}")
for i in range(device_count):
    print(f"  Device {i}: {torch.cuda.get_device_name(i)}")

WORK_DIR = Path("/kaggle/working/chagatai_sbd") if Path("/kaggle/working").exists() else Path("work/chagatai_sbd")
CHARLM_DIR = WORK_DIR / "charlm"
STANZA_DIR = WORK_DIR / "stanza"
MODEL_DIR = WORK_DIR / "models"
LOG_DIR = WORK_DIR / "logs"

for d in [CHARLM_DIR, STANZA_DIR, MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Copy bundled CharLM files from current dir or input notebooks
for f in ["chg_forward_charlm.pt", "chg_backward_charlm.pt", "chg_vocab.pt"]:
    if Path(f).exists():
        shutil.copy(f, CHARLM_DIR / f)
        print(f"Loaded bundled {f} ({Path(f).stat().st_size} bytes)")

input_root = Path("/kaggle/input")
for fname in ["chg_forward_charlm.pt", "chg_backward_charlm.pt", "chg_vocab.pt"]:
    cands = list(input_root.rglob(fname))
    if cands and not (CHARLM_DIR / fname).exists():
        shutil.copy(cands[0], CHARLM_DIR / fname)
        print(f"Copied {fname} from input: {cands[0]}")

for fname in ["train.txt", "train.toklabels", "dev.txt", "dev.toklabels", "test.txt", "test.toklabels", "mwt.json", "charlm_train.txt", "charlm_dev.txt"]:
    cands = list(input_root.rglob(fname))
    target_dir = CHARLM_DIR if "charlm" in fname else STANZA_DIR
    if cands and not (target_dir / fname).exists():
        shutil.copy(cands[0], target_dir / fname)
        print(f"Copied {fname} from input: {cands[0]}")

# Download & export dataset for Stanza if missing
if not (STANZA_DIR / "train.txt").exists():
    print("Exporting dataset: chagatai-project/chagatai-sbd (chagatai_uzs_uyghur_balanced)...")
    ds = load_dataset("chagatai-project/chagatai-sbd", "chagatai_uzs_uyghur_balanced")
    
    train_texts = [row["text"] for row in ds["train"] if row["text"].strip()]
    val_split = ds["validation"] if "validation" in ds else ds["dev"]
    val_texts = set(row["text"] for row in val_split if row["text"].strip())
    test_texts = set(row["text"] for row in ds["test"] if row["text"].strip())

    train_set = set(train_texts)
    assert train_set.isdisjoint(val_texts), "Data leak: validation text in train!"
    assert train_set.isdisjoint(test_texts), "Data leak: test text in train!"

    def stanza_text_and_labels(tokens, word_labels):
        chars, labels = [], []
        for token_index, (token, word_label) in enumerate(zip(tokens, word_labels)):
            if token_index:
                chars.append(" ")
                labels.append("0")
            for char_index, char in enumerate(token):
                chars.append(char)
                is_token_end = (char_index == len(token) - 1)
                if word_label and is_token_end:
                    labels.append("2")
                elif is_token_end:
                    labels.append("1")
                else:
                    labels.append("0")
        return "".join(chars), "".join(labels)

    def export_split(split_name, split_data):
        texts, labels = [], []
        for row in split_data:
            tokens = row.get("tokens") or row.get("words")
            word_labels = row.get("labels") or row.get("word_labels")
            t, l = stanza_text_and_labels(tokens, word_labels)
            texts.append(t)
            labels.append(l)
        (STANZA_DIR / f"{split_name}.txt").write_text("\n\n".join(texts) + "\n\n", encoding="utf-8")
        (STANZA_DIR / f"{split_name}.toklabels").write_text("\n\n".join(labels) + "\n\n", encoding="utf-8")

    export_split("train", ds["train"])
    export_split("dev", val_split)
    export_split("test", ds["test"])
    (STANZA_DIR / "mwt.json").write_text("{}", encoding="utf-8")
    print("Dataset successfully extracted with ZERO leakage!")

# Always ensure charlm_train.txt and charlm_dev.txt exist
if not (CHARLM_DIR / "charlm_train.txt").exists() or not (CHARLM_DIR / "charlm_dev.txt").exists():
    print("Creating charlm_train.txt and charlm_dev.txt from train.txt...")
    train_raw = (STANZA_DIR / "train.txt").read_text(encoding="utf-8").strip().split("\n\n")
    train_texts = [x.strip() for x in train_raw if x.strip()]
    split_idx = int(len(train_texts) * 0.95)
    (CHARLM_DIR / "charlm_train.txt").write_text("\n".join(train_texts[:split_idx]) + "\n", encoding="utf-8")
    (CHARLM_DIR / "charlm_dev.txt").write_text("\n".join(train_texts[split_idx:]) + "\n", encoding="utf-8")
    print(f"Created charlm_train.txt ({split_idx} seqs) and charlm_dev.txt ({len(train_texts) - split_idx} seqs)!")

if not (CHARLM_DIR / "chg_backward_charlm.pt").exists():
    print("Pretraining Backward CharLM on cuda:0...")
    import stanza.models.charlm as clm
    bwd_args = [
        '--train_file', str(CHARLM_DIR / 'charlm_train.txt'),
        '--eval_file', str(CHARLM_DIR / 'charlm_dev.txt'),
        '--direction', 'backward',
        '--shorthand', 'chg',
        '--save_dir', str(CHARLM_DIR),
        '--save_name', 'chg_backward_charlm.pt',
        '--char_emb_dim', '100',
        '--char_hidden_dim', '512',
        '--char_num_layers', '1',
        '--batch_size', '64',
        '--bptt_size', '250',
        '--epochs', '25',
        '--device', 'cuda:0' if torch.cuda.is_available() else 'cpu',
        '--cutoff', '5',
        '--checkpoint', 'False',
    ]
    clm.main(bwd_args)
    print("Backward CharLM ready!")


In [ ]:
# Write model_patch.py
patch_code = '''import torch
import torch.nn as nn
import torch.nn.functional as F
import stanza.models.tokenizer as tokenizer
import stanza.models.tokenization.trainer as trainer_mod
import stanza.models.tokenization.model as tm
from stanza.models.common.char_model import CharacterLanguageModelWordAdapter
from stanza.models.common.foundation_cache import load_charlm
from stanza.models.tokenization.vocab import Vocab
import logging

logger = logging.getLogger("stanza")

class FocalCrossEntropyLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, ignore_index=-1):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.ignore_index = ignore_index

    def forward(self, pred, target):
        ce_loss = F.cross_entropy(pred, target, weight=self.weight, ignore_index=self.ignore_index, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        mask = (target != self.ignore_index)
        return focal_loss[mask].mean()

def apply_model_patches():
    def patched_tok_init(self, args, nchars, emb_dim, hidden_dim, dropout, feat_dropout, foundation_cache=None):
        super(tm.Tokenizer, self).__init__()
        self.unsaved_modules = []
        self.args = args
        feat_dim = args["feat_dim"]
        self.embeddings = nn.Embedding(nchars, emb_dim, padding_idx=0)
        self.input_dim = emb_dim + feat_dim

        charmodels = []
        if args is not None and args.get("charlm_forward_file", None):
            charmodels.append(load_charlm(args["charlm_forward_file"], foundation_cache=foundation_cache))
        if args is not None and args.get("charlm_backward_file", None):
            charmodels.append(load_charlm(args["charlm_backward_file"], foundation_cache=foundation_cache))

        if charmodels:
            charmodel = CharacterLanguageModelWordAdapter(nn.ModuleList(charmodels))
            self.input_dim += charmodel.hidden_dim()
        else:
            charmodel = None
        self.add_unsaved_module("charmodel", charmodel)

        self.rnn = nn.LSTM(self.input_dim, hidden_dim, num_layers=self.args["rnn_layers"], bidirectional=True, batch_first=True, dropout=dropout if self.args["rnn_layers"] > 1 else 0)

        if self.args.get("conv_res", None) is not None:
            self.conv_res = nn.ModuleList()
            self.conv_sizes = [int(x) for x in self.args["conv_res"].split(",")]
            for si, size in enumerate(self.conv_sizes):
                l = nn.Conv1d(self.input_dim, hidden_dim * 2, size, padding=size//2, bias=self.args.get("hier_conv_res", False) or (si == 0))
                self.conv_res.append(l)
        else:
            self.conv_res = None

        self.tok_clf = nn.Linear(hidden_dim * 2, 1)
        self.sent_clf = nn.Linear(hidden_dim * 2, 1)
        if self.args.get("use_mwt", False):
            self.mwt_clf = nn.Linear(hidden_dim * 2, 1)

        if args.get("hierarchical", True):
            in_dim = hidden_dim * 2
            self.rnn2 = nn.LSTM(in_dim, hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
            self.tok_clf2 = nn.Linear(hidden_dim * 2, 1, bias=False)
            self.sent_clf2 = nn.Linear(hidden_dim * 2, 1, bias=False)
            if self.args.get("use_mwt", False):
                self.mwt_clf2 = nn.Linear(hidden_dim * 2, 1, bias=False)

        self.dropout = nn.Dropout(dropout)
        self.dropout_feat = nn.Dropout(feat_dropout)
        self.toknoise = nn.Dropout(self.args["tok_noise"])

    tm.Tokenizer.__init__ = patched_tok_init

    def patched_trainer_load(self, filename, args, foundation_cache):
        checkpoint = torch.load(filename, lambda storage, loc: storage, weights_only=True)
        self.args = checkpoint["config"]
        if args is not None and args.get("charlm_forward_file", None) is not None:
            if checkpoint["config"].get("charlm_forward_file") is None:
                self.args["charlm_forward_file"] = None
            else:
                self.args["charlm_forward_file"] = args["charlm_forward_file"]
        if args is not None and args.get("charlm_backward_file", None) is not None:
            if checkpoint["config"].get("charlm_backward_file") is None:
                self.args["charlm_backward_file"] = None
            else:
                self.args["charlm_backward_file"] = args["charlm_backward_file"]
        if self.args.get("use_mwt", None) is None:
            self.args["use_mwt"] = True
        self.model = tm.Tokenizer(self.args, self.args["vocab_size"], self.args["emb_dim"], self.args["hidden_dim"], dropout=self.args["dropout"], feat_dropout=self.args["feat_dropout"], foundation_cache=foundation_cache)
        self.model.load_state_dict(checkpoint["model"], strict=False)
        self.vocab = Vocab.load_state_dict(checkpoint["vocab"])
        self.lexicon = checkpoint["lexicon"]
        if self.lexicon is not None:
            self.lexicon = set(self.lexicon)
            self.dictionary = trainer_mod.create_dictionary(self.lexicon)
        else:
            self.dictionary = None

    trainer_mod.Trainer.load = patched_trainer_load

    orig_build_argparse = tokenizer.build_argparse
    def patched_build_argparse():
        parser = orig_build_argparse()
        parser.add_argument("--charlm_backward_file", type=str, default=None)
        return parser
    tokenizer.build_argparse = patched_build_argparse
'''
Path("model_patch.py").write_text(patch_code, encoding="utf-8")
print("model_patch.py written successfully!")

# Write train_worker.py
worker_code = '''import sys
import argparse
from pathlib import Path
import torch
import torch.nn as nn
import model_patch
import stanza.models.tokenizer as tokenizer
import stanza.models.tokenization.trainer as trainer_mod

model_patch.apply_model_patches()

parser = argparse.ArgumentParser()
parser.add_argument("--device", type=str, default="cuda:0")
parser.add_argument("--loss_type", type=str, choices=["weighted", "focal"], default="weighted")
parser.add_argument("--pos_weight", type=float, default=4.0)
parser.add_argument("--focal_gamma", type=float, default=2.0)
parser.add_argument("--save_name", type=str, required=True)
parser.add_argument("--work_dir", type=str, default="/kaggle/working/chagatai_sbd")
cli_args = parser.parse_args()

WORK_DIR = Path(cli_args.work_dir)
CHARLM_DIR = WORK_DIR / "charlm"
STANZA_DIR = WORK_DIR / "stanza"
MODEL_DIR = WORK_DIR / "models"

FWD_MODEL_PATH = CHARLM_DIR / "chg_forward_charlm.pt"
BWD_MODEL_PATH = CHARLM_DIR / "chg_backward_charlm.pt"

orig_trainer_init = trainer_mod.Trainer.__init__
def patched_trainer_init(self, args=None, vocab=None, lexicon=None, dictionary=None, model_file=None, device=None, foundation_cache=None):
    orig_trainer_init(self, args, vocab, lexicon, dictionary, model_file, device, foundation_cache)
    weights = torch.tensor([1.0, 1.0, cli_args.pos_weight], device=device)
    if cli_args.loss_type == "weighted":
        self.criterion = nn.CrossEntropyLoss(weight=weights, ignore_index=-1).to(device)
        print(f"[{cli_args.save_name}] Injected Weighted CrossEntropyLoss with weights {weights.tolist()} on {device}")
    elif cli_args.loss_type == "focal":
        self.criterion = model_patch.FocalCrossEntropyLoss(weight=weights, gamma=cli_args.focal_gamma, ignore_index=-1).to(device)
        print(f"[{cli_args.save_name}] Injected FocalCrossEntropyLoss (gamma={cli_args.focal_gamma}, weights={weights.tolist()}) on {device}")

trainer_mod.Trainer.__init__ = patched_trainer_init

train_args = [
    '--mode', 'train',
    '--txt_file', str(STANZA_DIR / 'train.txt'),
    '--label_file', str(STANZA_DIR / 'train.toklabels'),
    '--dev_txt_file', str(STANZA_DIR / 'dev.txt'),
    '--dev_label_file', str(STANZA_DIR / 'dev.toklabels'),
    '--mwt_json_file', str(STANZA_DIR / 'mwt.json'),
    '--lang', 'chg',
    '--shorthand', 'chg_balanced',
    '--save_dir', str(MODEL_DIR),
    '--save_name', cli_args.save_name,
    '--device', cli_args.device,
    '--max_seqlen', '1000',
    '--batch_size', '64',
    '--steps', '15000',
    '--eval_steps', '100',
    '--report_steps', '50',
    '--max_steps_before_stop', '1500',
    '--lr0', '0.002',
    '--weight_decay', '0.0',
    '--dropout', '0.33',
    '--unit_dropout', '0.0',
    '--emb_dim', '32',
    '--hidden_dim', '256',
    '--rnn_layers', '3',
    '--charlm',
    '--charlm_forward_file', str(FWD_MODEL_PATH),
    '--charlm_backward_file', str(BWD_MODEL_PATH),
    '--seed', '42' if 'weighted' in cli_args.save_name else '43'
]

print(f"[{cli_args.save_name}] Starting training on {cli_args.device}...")
tokenizer.main(train_args)
print(f"[{cli_args.save_name}] Finished successfully!")
'''
Path("train_worker.py").write_text(worker_code, encoding="utf-8")
print("train_worker.py written successfully!")


In [ ]:
import subprocess
import time
import sys
import os

log0_path = LOG_DIR / "worker_gpu0.log"
log1_path = LOG_DIR / "worker_gpu1.log"

env0 = os.environ.copy()
env0["CUDA_VISIBLE_DEVICES"] = "0"

env1 = os.environ.copy()
env1["CUDA_VISIBLE_DEVICES"] = "1"

cmd0 = [
    sys.executable, "train_worker.py",
    "--device", "cuda:0",
    "--loss_type", "weighted",
    "--pos_weight", "4.0",
    "--save_name", "chg_bicharlm_weighted_wide3.pt",
    "--work_dir", str(WORK_DIR)
]

cmd1 = [
    sys.executable, "train_worker.py",
    "--device", "cuda:0",
    "--loss_type", "focal",
    "--pos_weight", "4.0",
    "--focal_gamma", "2.0",
    "--save_name", "chg_bicharlm_focal_wide3.pt",
    "--work_dir", str(WORK_DIR)
]

print("=== LAUNCHING CONCURRENT TRAINING ON BOTH GPUS ===")
print("  Physical GPU 0 -> Bidirectional CharLM + Weighted Loss (pos_weight=4.0)")
print("  Physical GPU 1 -> Bidirectional CharLM + Focal Loss (pos_weight=4.0, gamma=2.0)")

f0 = open(log0_path, "w", encoding="utf-8")
f1 = open(log1_path, "w", encoding="utf-8")

p0 = subprocess.Popen(cmd0, env=env0, stdout=f0, stderr=subprocess.STDOUT)
p1 = subprocess.Popen(cmd1, env=env1, stdout=f1, stderr=subprocess.STDOUT)

start_time = time.time()
print(f"Workers spawned with PIDs: GPU 0 -> {p0.pid}, GPU 1 -> {p1.pid}")

# Monitor both processes in real time
while p0.poll() is None or p1.poll() is None:
    elapsed = int(time.time() - start_time)
    s0 = "RUNNING" if p0.poll() is None else f"DONE (code {p0.returncode})"
    s1 = "RUNNING" if p1.poll() is None else f"DONE (code {p1.returncode})"
    print(f"[{elapsed}s elapsed] GPU 0: {s0} | GPU 1: {s1}")
    time.sleep(30)

f0.close()
f1.close()

total_time = int(time.time() - start_time)
print(f"\nBoth GPU workers completed in {total_time} seconds ({total_time/60:.1f} minutes)!")

if p0.returncode != 0:
    print("--- GPU 0 FAILURE LOG ---")
    print(log0_path.read_text(encoding="utf-8", errors="replace")[-2000:])
if p1.returncode != 0:
    print("--- GPU 1 FAILURE LOG ---")
    print(log1_path.read_text(encoding="utf-8", errors="replace")[-2000:])

assert p0.returncode == 0, f"GPU 0 failed with code {p0.returncode}. Check {log0_path}"
assert p1.returncode == 0, f"GPU 1 failed with code {p1.returncode}. Check {log1_path}"


In [ ]:
print("=== EVALUATION AND ENSEMBLING ===")
import numpy as np
import torch
from torch.utils.data import DataLoader as TorchDataLoader
from stanza.models.tokenization.data import TokenizationDataset, SortedDataset
import stanza.models.tokenizer as tokenizer
import stanza.models.tokenization.trainer as trainer_mod

# Apply patches to evaluate bidirectional CharLM models properly
import model_patch
model_patch.apply_model_patches()

def get_probs_and_gold(model_path, txt_path, label_path):
    eval_args = tokenizer.parse_args([
        '--mode', 'predict',
        '--txt_file', str(txt_path),
        '--label_file', str(label_path),
        '--mwt_json_file', str(STANZA_DIR / 'mwt.json'),
        '--lang', 'chg',
        '--shorthand', 'chg_balanced',
        '--device', 'cuda:0' if torch.cuda.is_available() else 'cpu',
        '--max_seqlen', '1000',
        '--batch_size', '64',
        '--charlm',
        '--charlm_forward_file', str(CHARLM_DIR / 'chg_forward_charlm.pt'),
        '--charlm_backward_file', str(CHARLM_DIR / 'chg_backward_charlm.pt'),
    ])
    trainer = trainer_mod.Trainer(args=eval_args, model_file=str(model_path), device='cuda:0' if torch.cuda.is_available() else 'cpu', foundation_cache=None)
    for k, v in trainer.args.items():
        if not k.endswith('_file') and k not in {'device', 'mode', 'save_dir', 'load_name', 'save_name'}:
            eval_args[k] = v

    dataset = TokenizationDataset(
        eval_args,
        input_files={'txt': str(txt_path), 'label': str(label_path)},
        vocab=trainer.vocab,
        evaluation=True,
        dictionary=trainer.dictionary,
    )
    sorted_data = SortedDataset(dataset)
    dataloader = TorchDataLoader(sorted_data, batch_size=64, collate_fn=sorted_data.collate)

    all_lps = []
    all_raw = []
    for batch_idx, batch in enumerate(dataloader):
        num_sentences = len(batch[3])
        for paragraph in batch[3]:
            all_raw.append(list(paragraph))
        log_probs = trainer.predict(batch)
        for par_idx in range(num_sentences):
            offset = batch_idx * 64 + par_idx
            raw = all_raw[offset]
            par_len = raw.index('<PAD>') if '<PAD>' in raw else len(raw)
            all_lps.append(log_probs[par_idx, :par_len, :])

    all_lps = sorted_data.unsort(all_lps)
    gold_labels = dataset.labels()
    
    probs_list = []
    for lp in all_lps:
        probs = np.exp(lp)
        p_tok = probs[:, 1] + probs[:, 2]
        p_inside = probs[:, 0]
        p_sent_cond = np.zeros(len(probs))
        tok_mask = p_tok > 1e-9
        p_sent_cond[tok_mask] = probs[tok_mask, 2] / p_tok[tok_mask]
        probs_list.append((p_tok > p_inside, p_sent_cond))
    return probs_list, gold_labels

def eval_preds(probs_list, gold_labels, tau=0.50):
    pred_boundaries = []
    gold_boundaries = []
    for (is_tok, p_sent_cond), g in zip(probs_list, gold_labels):
        is_eos = np.logical_and(is_tok, p_sent_cond >= tau)
        is_boundary = np.zeros(len(p_sent_cond), dtype=int)
        is_boundary[is_tok] = 1
        is_boundary[is_eos] = 2
        if is_boundary[-1] < 2:
            is_boundary[-1] = 2
        pred_boundaries.append(is_boundary == 2)
        gold_boundaries.append(np.isin(g, [2, 4]))

    p_sent = np.concatenate(pred_boundaries)
    g_sent = np.concatenate(gold_boundaries)
    tp = int(np.logical_and(p_sent, g_sent).sum())
    fp = int(np.logical_and(p_sent, ~g_sent).sum())
    fn = int(np.logical_and(~p_sent, g_sent).sum())
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    exact = sum(np.array_equal(p, g) for p, g in zip(pred_boundaries, gold_boundaries))
    return {'prec': prec, 'rec': rec, 'f1': f1, 'exact_count': exact, 'exact_pct': exact / len(pred_boundaries), 'tp': tp, 'fp': fp, 'fn': fn}

def find_best_tau(dev_probs, dev_gold):
    best_tau, best_f1 = 0.50, -1.0
    for tau in np.arange(0.15, 0.65, 0.02):
        res = eval_preds(dev_probs, dev_gold, tau=tau)
        if res['f1'] > best_f1:
            best_f1 = res['f1']
            best_tau = tau
    return best_tau, best_f1

m0_path = MODEL_DIR / "chg_bicharlm_weighted_wide3.pt"
m1_path = MODEL_DIR / "chg_bicharlm_focal_wide3.pt"

print("Extracting probabilities on Dev and Test...")
dev_p0, dev_g = get_probs_and_gold(m0_path, STANZA_DIR / "dev.txt", STANZA_DIR / "dev.toklabels")
test_p0, test_g = get_probs_and_gold(m0_path, STANZA_DIR / "test.txt", STANZA_DIR / "test.toklabels")

dev_p1, _ = get_probs_and_gold(m1_path, STANZA_DIR / "dev.txt", STANZA_DIR / "dev.toklabels")
test_p1, _ = get_probs_and_gold(m1_path, STANZA_DIR / "test.txt", STANZA_DIR / "test.toklabels")

# Ensemble blend
dev_blend = []
for i in range(len(dev_p0)):
    dev_blend.append((dev_p0[i][0], 0.5 * dev_p0[i][1] + 0.5 * dev_p1[i][1]))

test_blend = []
for i in range(len(test_p0)):
    test_blend.append((test_p0[i][0], 0.5 * test_p0[i][1] + 0.5 * test_p1[i][1]))

tau0, _ = find_best_tau(dev_p0, dev_g)
tau1, _ = find_best_tau(dev_p1, dev_g)
tau_ens, _ = find_best_tau(dev_blend, dev_g)

res0 = eval_preds(test_p0, test_g, tau=tau0)
res1 = eval_preds(test_p1, test_g, tau=tau1)
res_ens = eval_preds(test_blend, test_g, tau=tau_ens)

print("\n=================================================================")
print("FINAL TEST METRICS (DUAL-GPU TASK PARALLELISM - 295 SAMPLES)")
print("=================================================================")
print(f"1. Model 0 (BiCharLM + Weighted Loss, tau={tau0:.2f}):")
print(f"   F1: {res0['f1']*100:.2f}% | Prec: {res0['prec']*100:.2f}% | Rec: {res0['rec']*100:.2f}% | Exact: {res0['exact_pct']*100:.2f}% ({res0['exact_count']}/295) | TP={res0['tp']} FP={res0['fp']} FN={res0['fn']}")

print(f"\n2. Model 1 (BiCharLM + Focal Loss, tau={tau1:.2f}):")
print(f"   F1: {res1['f1']*100:.2f}% | Prec: {res1['prec']*100:.2f}% | Rec: {res1['rec']*100:.2f}% | Exact: {res1['exact_pct']*100:.2f}% ({res1['exact_count']}/295) | TP={res1['tp']} FP={res1['fp']} FN={res1['fn']}")

print(f"\n3. Dual-GPU Ensemble (Model 0 + Model 1, tau={tau_ens:.2f}):")
print(f"   F1: {res_ens['f1']*100:.2f}% | Prec: {res_ens['prec']*100:.2f}% | Rec: {res_ens['rec']*100:.2f}% | Exact: {res_ens['exact_pct']*100:.2f}% ({res_ens['exact_count']}/295) | TP={res_ens['tp']} FP={res_ens['fp']} FN={res_ens['fn']}")
print("=================================================================")

metrics_out = WORK_DIR / "dual_gpu_parallel_test_metrics.json"
with open(metrics_out, "w", encoding="utf-8") as f:
    json.dump({
        "model0_weighted": {"tau": tau0, "metrics": res0},
        "model1_focal": {"tau": tau1, "metrics": res1},
        "ensemble_dual": {"tau": tau_ens, "metrics": res_ens}
    }, f, indent=2)
print(f"Saved metrics to {metrics_out}")
